In [1]:
"""Utilities for Quad4FHE plaintext experiments.

This module is intentionally small and dependency-light so it can be copied next
into any single-dataset notebook/script. It provides two things:

1. A tee context manager that writes all notebook/script stdout/stderr to a log
   file while still showing it on screen.
2. Direct JSON export from quad4fhe.ReplacementResult objects, avoiding fragile
   regex parsing of copied notebook output.
"""

from __future__ import annotations

import contextlib
import json
import math
import sys
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, Optional


class _TeeStream:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()

    def flush(self):
        for stream in self.streams:
            stream.flush()

    def isatty(self):
        return any(getattr(stream, "isatty", lambda: False)() for stream in self.streams)


@contextlib.contextmanager
def tee_stdout_stderr(log_path: str | Path):
    """Duplicate stdout/stderr to ``log_path`` and the current console.

    Use around the whole experiment:

        with tee_stdout_stderr("results/my_run.txt"):
            main()
    """
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    old_stdout, old_stderr = sys.stdout, sys.stderr
    with log_path.open("w", encoding="utf-8", buffering=1) as fh:
        sys.stdout = _TeeStream(old_stdout, fh)
        sys.stderr = _TeeStream(old_stderr, fh)
        try:
            print(f"[autosave] stdout/stderr log -> {log_path}")
            yield log_path
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


def to_jsonable(obj: Any) -> Any:
    """Convert common scientific Python objects to strict JSON values."""
    # Local imports keep this helper usable even in minimal environments.
    try:
        import numpy as np
    except Exception:  # pragma: no cover
        np = None
    try:
        import pandas as pd
    except Exception:  # pragma: no cover
        pd = None
    try:
        import torch
    except Exception:  # pragma: no cover
        torch = None

    if obj is None or isinstance(obj, (str, bool, int)):
        return obj

    if isinstance(obj, float):
        return obj if math.isfinite(obj) else None

    if np is not None:
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            value = float(obj)
            return value if math.isfinite(value) else None
        if isinstance(obj, np.bool_):
            return bool(obj)
        if isinstance(obj, np.ndarray):
            return to_jsonable(obj.tolist())

    if torch is not None and hasattr(torch, "is_tensor") and torch.is_tensor(obj):
        return to_jsonable(obj.detach().cpu().numpy())

    if isinstance(obj, Path):
        return str(obj)

    if pd is not None:
        if isinstance(obj, pd.DataFrame):
            return to_jsonable(obj.to_dict(orient="records"))
        if isinstance(obj, pd.Series):
            return to_jsonable(obj.to_dict())

    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_jsonable(v) for v in obj]

    # Last resort: preserve readable values without breaking json.dump.
    return str(obj)


def save_json(obj: Dict[str, Any], path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        json.dump(to_jsonable(obj), fh, indent=2, ensure_ascii=False, allow_nan=False)
    print(f"[autosave] JSON -> {path}")
    return path


def save_csv(rows: Iterable[Dict[str, Any]], path: str | Path) -> Path:
    import pandas as pd
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(list(rows)).to_csv(path, index=False)
    print(f"[autosave] CSV -> {path}")
    return path


def dataframe_records(df: Any) -> list:
    if df is None:
        return []
    try:
        return to_jsonable(df.to_dict(orient="records"))
    except Exception:
        return []


def dataframe_test_by_method(df: Any) -> Dict[str, Dict[str, Any]]:
    """Return {method: metrics} for rows with Split == 'test'."""
    out: Dict[str, Dict[str, Any]] = {}
    if df is None:
        return out
    try:
        sub = df[df["Split"] == "test"]
        for _, row in sub.iterrows():
            method = str(row.get("Method"))
            metrics = {str(k): to_jsonable(v) for k, v in row.to_dict().items()
                       if k not in ("Method", "Split")}
            out[method] = metrics
    except Exception:
        return out
    return out


def _metric_from_table(table: Dict[str, Dict[str, Any]], method: str, *keys: str) -> Any:
    row = table.get(method, {})
    for key in keys:
        if key in row:
            return row[key]
    return None


def quad_report_diagnostics(result: Any, split: str, n_expected: Optional[int] = None) -> Dict[str, Any]:
    """Extract agreement/mismatch diagnostics for fit/calibration or test split."""
    attr_candidates = []
    if split in ("fit", "calib", "calibration"):
        attr_candidates = ["fit_diagnostics", "calib_diagnostics", "calibration_diagnostics"]
        split_label = "fit"
    else:
        attr_candidates = ["test_diagnostics"]
        split_label = "test"

    diag = None
    for attr in attr_candidates:
        value = getattr(result, attr, None)
        if value is not None:
            diag = dict(value)
            break

    if diag is None:
        diag = {}
        df = getattr(result, "report_vs_pseudo", None)
        if df is not None:
            try:
                row = df[(df["Method"] == "Quad4FHE") & (df["Split"] == split_label)]
                if len(row) > 0:
                    diag["agreement"] = float(row.iloc[0]["Agreement"])
            except Exception:
                pass

    n_value = diag.get("n", diag.get("calib_n", diag.get("n_samples", n_expected)))
    agreement = diag.get("agreement", diag.get("calib_agreement", diag.get("fit_agreement")))
    mismatch = diag.get("mismatch_count", diag.get("calib_mismatch_count", diag.get("fit_mismatch_count")))

    if mismatch is None and agreement is not None and n_value is not None:
        mismatch = int(round((1.0 - float(agreement)) * int(n_value)))

    exact = diag.get("exact_preserved", diag.get("exact_preserved_on_calib", diag.get("fit_exact_preserved")))
    if exact is None and mismatch is not None:
        exact = bool(int(mismatch) == 0)

    return {
        "split": split_label,
        "n": to_jsonable(n_value),
        "agreement": to_jsonable(agreement),
        "mismatch_count": to_jsonable(mismatch),
        "exact_preserved": to_jsonable(exact),
    }


def quad_solver_diagnostics(result: Any) -> Dict[str, Any]:
    return to_jsonable(dict(getattr(result, "solver_diagnostics", None) or {}))


def build_quad4fhe_run_record(
    *,
    result: Any,
    key: str,
    hidden_dim: Optional[int],
    fit_n: Optional[int],
    test_n: Optional[int],
    pool_fraction: Optional[float] = None,
    rep_fit_size: Optional[int] = None,
    extra: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """Build a JSON-serializable record for one Quad4FHE run."""
    fit_diag = quad_report_diagnostics(result, "fit", n_expected=fit_n)
    test_diag = quad_report_diagnostics(result, "test", n_expected=test_n)
    solver_diag = quad_solver_diagnostics(result)

    report_truth_test = dataframe_test_by_method(getattr(result, "report_vs_truth", None))
    report_pseudo_test = dataframe_test_by_method(getattr(result, "report_vs_pseudo", None))

    calib_agreement = fit_diag.get("agreement")
    test_agreement = test_diag.get("agreement")
    gap = None
    if calib_agreement is not None and test_agreement is not None:
        gap = float(calib_agreement) - float(test_agreement)

    # Common keys requested by reviewers. Keep all solver diagnostics too.
    common_solver_keys = [
        "num_pairwise_constraints",
        "min_pairwise_margin",
        "normalized_min_pairwise_margin",
        "slack_positive_count",
        "sum_slack",
        "mean_slack",
        "max_slack",
        "pairwise_slack_positive_count",
        "pairwise_sum_slack",
        "pairwise_mean_slack",
        "pairwise_max_slack",
        "selected_C",
        "soft_C_grid",
        "soft_trace",
        "selected_mu",
        "mu_grid",
        "mu_p",
        "mu_n",
    ]

    quad = {
        "alpha": getattr(result, "alpha", None),
        "beta": getattr(result, "beta", None),
        "eta": getattr(result, "eta", None),
        "threshold": getattr(result, "threshold", None),
        "zero_threshold_realized": getattr(result, "zero_threshold_realized", None),
        "method_used": getattr(result, "method_used", None),
        "hard_feasible": getattr(result, "feasible", None),
        "empirical_margin": getattr(result, "empirical_margin", None),
        "normalized_margin": getattr(result, "normalized_margin", None),
        "quant_decimals": getattr(result, "quant_decimals", None),
        "constraint_version": getattr(result, "constraint_version", None),
        "he_artifact_dir": str(getattr(result, "he_export_dir", None)) if getattr(result, "he_export_dir", None) else None,
        "calib_n": fit_diag.get("n"),
        "calib_agreement": calib_agreement,
        "calib_mismatch_count": fit_diag.get("mismatch_count"),
        "exact_preserved_on_calib": fit_diag.get("exact_preserved"),
        "test_n": test_diag.get("n"),
        "test_agreement": test_agreement,
        "test_mismatch_count": test_diag.get("mismatch_count"),
        "calib_test_agreement_gap": gap,
        "test_top1_acc": _metric_from_table(report_truth_test, "Quad4FHE", "ACC", "Top1", "Top-1"),
        "test_top5_acc": _metric_from_table(report_truth_test, "Quad4FHE", "Top5", "Top-5"),
        "test_macro_f1": _metric_from_table(report_truth_test, "Quad4FHE", "MacroF1", "F1"),
        "solver_diagnostics": solver_diag,
    }
    for k in common_solver_keys:
        quad[k] = solver_diag.get(k)

    # Fill a few derived diagnostics that are convenient for tables.
    if quad.get("pairwise_mean_slack") is None:
        denom = quad.get("num_pairwise_constraints")
        num = quad.get("pairwise_sum_slack")
        try:
            if denom not in (None, 0) and num is not None:
                quad["pairwise_mean_slack"] = float(num) / float(denom)
        except Exception:
            pass
    if quad.get("selected_mu") is None:
        method = quad.get("method_used")
        try:
            if isinstance(method, str) and method.startswith("rch") and quad.get("mu_p") == quad.get("mu_n"):
                quad["selected_mu"] = quad.get("mu_p")
        except Exception:
            pass

    original = {
        "test_top1_acc": _metric_from_table(report_truth_test, "Original", "ACC", "Top1", "Top-1"),
        "test_top5_acc": _metric_from_table(report_truth_test, "Original", "Top5", "Top-5"),
    }

    record = {
        "key": key,
        "hidden_dim": hidden_dim,
        "pool_fraction": pool_fraction,
        "rep_fit_size": rep_fit_size,
        "original": original,
        "quad4fhe": quad,
        "report_vs_ground_truth_test": report_truth_test,
        "report_vs_pseudo_labels_test": report_pseudo_test,
        "report_vs_ground_truth_records": dataframe_records(getattr(result, "report_vs_truth", None)),
        "report_vs_pseudo_labels_records": dataframe_records(getattr(result, "report_vs_pseudo", None)),
    }
    if extra:
        record.update(to_jsonable(extra))
    return to_jsonable(record)


def make_experiment_payload(
    *,
    dataset: str,
    experiment: str,
    seed: int,
    dataset_info: Dict[str, Any],
    config: Dict[str, Any],
    source_script: Optional[str] = None,
    log_file: Optional[str | Path] = None,
) -> Dict[str, Any]:
    return {
        "dataset": dataset,
        "experiment": experiment,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "source_script": source_script,
        "log_file": str(log_file) if log_file is not None else None,
        "seed": seed,
        "dataset_info": to_jsonable(dataset_info),
        "config": to_jsonable(config),
        "runs": [],
    }


def write_combined_dataset_json(dataset: str, root_dir: str | Path = "results",
                                output_path: Optional[str | Path] = None) -> Optional[Path]:
    """Merge autosaved fulltrain/smallpool JSONs into one dataset-level JSON.

    This keeps compatibility with the older `<dataset>_results.json` convention.
    Missing halves are skipped, so it is safe to call after either notebook.
    """
    root = Path(root_dir) / dataset
    if output_path is None:
        output_path = root / f"{dataset}_results.json"
    output_path = Path(output_path)

    combined: Dict[str, Any] = {"dataset": dataset, "created_at": datetime.now().isoformat(timespec="seconds")}
    found = False
    for exp in ("fulltrain", "smallpool"):
        p = root / exp / f"{dataset}_{exp}_results.json"
        if not p.exists():
            continue
        with p.open("r", encoding="utf-8") as fh:
            block = json.load(fh)
        combined[exp] = {
            "source_json": str(p),
            "source_script": block.get("source_script"),
            "log_file": block.get("log_file"),
            "dataset_info": block.get("dataset_info", {}),
            "config": block.get("config", {}),
            "runs": block.get("runs", []),
        }
        if exp == "fulltrain":
            combined[exp]["cross_hidden_dim_summary"] = block.get("cross_hidden_dim_summary", [])
        if exp == "smallpool":
            combined[exp]["cross_pool_tables"] = block.get("cross_pool_tables", {})
            combined[exp]["meta_table"] = block.get("meta_table", [])
        found = True

    if not found:
        print(f"[autosave] no fulltrain/smallpool JSON found under {root}; combined JSON not written")
        return None
    return save_json(combined, output_path)


import traceback


In [2]:
import os
import gc
import sys
import copy
import time
import random
import warnings
import faulthandler
from typing import Dict, List, Tuple
from pathlib import Path

faulthandler.enable()
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset

from PIL import Image
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from transformers import AutoImageProcessor, AutoModel

try:
    from datasets import load_dataset as hf_load_dataset
    HAS_HF_DATASETS = True
except Exception:
    hf_load_dataset = None
    HAS_HF_DATASETS = False

import quad4fhe


# ============================================================
# Configuration
# ============================================================
SEED = 2026
NUM_CLASSES = 196                                # Stanford Cars

HIDDEN_DIM = 256

# 4-way split fractions (must sum to 1.0)
TRAIN_SIZE     = 0.50
VAL_SIZE       = 0.10
TEST_SIZE      = 0.20
POOL_MAX_SIZE  = 0.20

IMG_RESIZE = 224                                 # Resize all Stanford Cars images

# Fraction of TOTAL dataset used as rep_fit for each pool size.
# For Stanford Cars (~16k samples, 196 classes):
#   1% -> ~160  samples  (< 196 classes; fallback to non-stratified; likely SKIPPED by MIN_SAMPLES_PER_CLASS)
#   5% -> ~820  samples  (~4 samples/class on average; usually OK)
#  10% -> ~1600 samples  (~8 samples/class on average)
#  20% -> ~3200 samples  (~16 samples/class, == full POOL_MAX)
SURROGATE_FRACTIONS = [0.01, 0.05, 0.10, 0.20]

MIN_POOL_SAMPLES = 50
MIN_SAMPLES_PER_CLASS = 2

EPOCHS = 100
LR = 1e-3
BATCH_SIZE_TRAIN = 256
BATCH_SIZE_EVAL = 1024
WEIGHT_DECAY = 1e-5
PATIENCE = 10
PRINT_EVERY = 5

# --- Paths ---
STANFORDCARS_DATA_ROOT = "./data_StanfordCars"
DINOV2_LOCAL_PATH      = "./dinov2-base"
FEATURE_CACHE          = "./data/dinov2_base_stanfordcars_cls_features.npz"
DATASET_NAME = "Dinov2_StanfordCars"
EXPERIMENT_NAME = "smallpool"
OUTPUT_DIR = Path("..") / "results" / DATASET_NAME / EXPERIMENT_NAME
LOG_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_result.txt"
JSON_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_results.json"
META_CSV_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_meta.csv"

BATCH_SIZE_FEATURE = 64
NUM_WORKERS        = 0
USE_AMP            = True

BASELINES = ["square", "ls_poly_2", "ls_poly_3", "ls_poly_5", "ls_poly_7",
             "remez_2", "remez_3", "remez_5", "remez_7",
             "ola", "precise_a11"]

# Stanford Cars class names (populated at runtime)
STANFORDCARS_CLASS_NAMES: Dict[int, str] = {}


# ============================================================
# Utilities
# ============================================================
def separator(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)
    sys.stdout.flush()


def sub_separator(title):
    print("\n" + "-" * 78)
    print(title)
    print("-" * 78)
    sys.stdout.flush()


def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


def topk_accuracy(logits, y_true, k=5):
    topk = np.argpartition(-logits, kth=k-1, axis=1)[:, :k]
    return float(np.mean([y_true[i] in topk[i] for i in range(len(y_true))]))


# ============================================================
# Stanford Cars loading + corruption splits + 4-way split
# ============================================================
def load_stanford_cars_merged(root=STANFORDCARS_DATA_ROOT) -> Tuple[np.ndarray, np.ndarray]:
    """Load Stanford Cars (train + test merged) from local parquet files."""
    import glob as _glob
    global STANFORDCARS_CLASS_NAMES

    if not HAS_HF_DATASETS:
        raise ImportError(
            "The 'datasets' library is required to load Stanford Cars parquet files.\n"
            "Install with:  pip install datasets"
        )

    print(f"Loading Stanford Cars from local parquet files in: {root}")
    train_files = sorted(_glob.glob(os.path.join(root, "train-*.parquet")))
    test_files  = sorted(_glob.glob(os.path.join(root, "test-*.parquet")))

    if not train_files:
        raise FileNotFoundError(f"No train parquet files found in {root}.")
    if not test_files:
        raise FileNotFoundError(f"No test parquet files found in {root}.")

    print(f"  Train files: {[os.path.basename(f) for f in train_files]}")
    print(f"  Test  files: {[os.path.basename(f) for f in test_files]}")

    data_files = {"train": train_files, "test": test_files}
    ds = hf_load_dataset("parquet", data_files=data_files)

    for split_name in ["train", "test"]:
        if split_name in ds:
            feat = ds[split_name].features.get("label")
            if feat is not None and hasattr(feat, "names"):
                STANFORDCARS_CLASS_NAMES = {i: name for i, name in enumerate(feat.names)}
                break

    images_list: List[np.ndarray] = []
    labels_list: List[int] = []

    for split_name in ["train", "test"]:
        if split_name not in ds:
            print(f"  WARNING: split '{split_name}' not found, skipping")
            continue
        split_ds = ds[split_name]
        n = len(split_ds)
        for idx in range(n):
            example = split_ds[idx]
            img = example["image"]
            label = int(example["label"])
            img = img.convert("RGB").resize((IMG_RESIZE, IMG_RESIZE), Image.BILINEAR)
            images_list.append(np.asarray(img, dtype=np.uint8))
            labels_list.append(label)
            if (idx + 1) == 1 or (idx + 1) % 2000 == 0 or (idx + 1) == n:
                print(f"  [{split_name}] loaded {idx + 1}/{n}")
        print(f"  Loaded {n} {split_name} images")

    if len(images_list) == 0:
        raise RuntimeError("No images loaded. Please check the parquet files.")

    X = np.stack(images_list, axis=0)
    y = np.asarray(labels_list, dtype=np.int64)
    print(f"  Total images (train + test): {len(X)}, shape: {X.shape}, classes: {len(np.unique(y))}")
    return X, y


def load_corruption_splits(root=STANFORDCARS_DATA_ROOT) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    """Discover and load corruption parquet files from *root*.

    Files are grouped by corruption name (prefix before ``-00xxx-of-00xxx.parquet``).
    Files starting with ``train-`` or ``test-`` are skipped.

    Returns dict  {corruption_name: (X_uint8, y_int64)}.
    """
    import glob as _glob
    import re as _re

    if not HAS_HF_DATASETS:
        return {}

    all_pq = sorted(_glob.glob(os.path.join(root, "*.parquet")))
    skip_prefixes = ("train-", "test-")
    corr_files: Dict[str, List[str]] = {}
    for fpath in all_pq:
        fname = os.path.basename(fpath)
        if any(fname.startswith(sp) for sp in skip_prefixes):
            continue
        m = _re.match(r"^(.+?)-\d{5}-of-\d{5}\.parquet$", fname)
        if m:
            corr_name = m.group(1)
            corr_files.setdefault(corr_name, []).append(fpath)

    if not corr_files:
        print("  No corruption parquet files found.")
        return {}

    print(f"  Found {len(corr_files)} corruption splits: {list(corr_files.keys())}")

    result: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
    for corr_name, files in sorted(corr_files.items()):
        files = sorted(files)
        print(f"  Loading corruption '{corr_name}' ({len(files)} file(s)) ...")
        ds = hf_load_dataset("parquet", data_files={"data": files})["data"]
        n = len(ds)
        imgs: List[np.ndarray] = []
        labels: List[int] = []
        for idx in range(n):
            ex = ds[idx]
            img = ex["image"].convert("RGB").resize((IMG_RESIZE, IMG_RESIZE), Image.BILINEAR)
            imgs.append(np.asarray(img, dtype=np.uint8))
            labels.append(int(ex["label"]))
        X_c = np.stack(imgs, axis=0)
        y_c = np.asarray(labels, dtype=np.int64)
        result[corr_name] = (X_c, y_c)
        print(f"    -> {corr_name}: {n} images, {len(np.unique(y_c))} classes")

    return result


def split_4way_indices(y, train_size=TRAIN_SIZE, val_size=VAL_SIZE,
                        test_size=TEST_SIZE, pool_size=POOL_MAX_SIZE, seed=SEED):
    """Split into disjoint subsets summing to 1.0: train / val / test / pool_max."""
    total = train_size + val_size + test_size + pool_size
    if abs(total - 1.0) > 1e-6:
        raise ValueError(f"Split fractions must sum to 1.0, got {total:.6f}")

    idx_all = np.arange(len(y))
    idx_rest, idx_test = train_test_split(idx_all, test_size=test_size, random_state=seed, stratify=y)
    pool_frac_of_rest = pool_size / (1.0 - test_size)
    idx_rest2, idx_pool_max = train_test_split(
        idx_rest, test_size=pool_frac_of_rest, random_state=seed + 1, stratify=y[idx_rest])
    val_frac_of_rest2 = val_size / (train_size + val_size)
    idx_train, idx_val = train_test_split(
        idx_rest2, test_size=val_frac_of_rest2, random_state=seed + 2, stratify=y[idx_rest2])

    return {
        "train":    np.sort(idx_train),
        "val":      np.sort(idx_val),
        "pool_max": np.sort(idx_pool_max),
        "test":     np.sort(idx_test),
    }


def subsample_pool(y_pool, fraction_of_total, total_size, seed):
    """Stratified subsample of size round(fraction*total) from the pool.

    Falls back to non-stratified random sampling when target_size < num_classes.
    """
    target = int(round(total_size * fraction_of_total))
    target = max(target, 1)
    n_pool = len(y_pool)
    if target >= n_pool:
        return np.arange(n_pool, dtype=np.int64)

    idx = np.arange(n_pool, dtype=np.int64)
    n_classes_present = len(np.unique(y_pool))
    can_stratify = target >= n_classes_present

    if can_stratify:
        try:
            sub_idx, _ = train_test_split(idx, train_size=target, random_state=seed, stratify=y_pool)
            return np.sort(sub_idx)
        except ValueError as e:
            print(f"  [subsample] stratified split failed ({e}); "
                  f"falling back to non-stratified sampling.")

    print(
        f"  [subsample] Non-stratified sampling: target={target} < "
        f"num_classes_in_pool={n_classes_present}; some classes will be absent."
    )
    rng = np.random.default_rng(seed)
    sub_idx = rng.choice(idx, size=target, replace=False)
    return np.sort(sub_idx)


# ============================================================
# DINOv2 feature extraction
# ============================================================
class ArrayImageDataset(Dataset):
    def __init__(self, X, y):
        self.X = X; self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return Image.fromarray(self.X[idx]), int(self.y[idx])


def make_image_collate(processor):
    def _collate(batch):
        images = [b[0] for b in batch]
        labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
        proc = processor(images=images, return_tensors="pt")
        return proc["pixel_values"], labels
    return _collate


@torch.no_grad()
def extract_dinov2_features(X, y, model_path, device,
                             processor=None, backbone=None):
    owns_backbone = backbone is None
    if owns_backbone:
        print(f"Extracting DINOv2 features from {model_path} ...")
        processor = AutoImageProcessor.from_pretrained(model_path)
        backbone = AutoModel.from_pretrained(model_path).to(device); backbone.eval()

    ds = ArrayImageDataset(X, y)
    loader = DataLoader(ds, batch_size=BATCH_SIZE_FEATURE, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=(device.type == "cuda" and NUM_WORKERS > 0),
                        collate_fn=make_image_collate(processor))

    feats_all, labels_all = [], []
    use_amp = USE_AMP and device.type == "cuda"
    for step, (pixel_values, labels) in enumerate(loader, start=1):
        pixel_values = pixel_values.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=use_amp,
                             dtype=torch.float16 if device.type == "cuda" else torch.float32):
            outputs = backbone(pixel_values=pixel_values)
            cls = outputs.last_hidden_state[:, 0, :]
        feats_all.append(cls.detach().cpu().float().numpy())
        labels_all.append(labels.numpy())
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"  Feature batch {step:>4d}/{len(loader)}")

    feats = np.concatenate(feats_all, axis=0).astype(np.float32)
    labels = np.concatenate(labels_all, axis=0).astype(np.int64)

    del loader, ds
    if owns_backbone:
        del backbone, processor
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        print("  DINOv2 backbone released from memory.")
    return feats, labels


def ensure_feature_cache(X, y, model_path, cache_path, device,
                         processor=None, backbone=None):
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    if os.path.exists(cache_path):
        print(f"Loading cached features: {cache_path}")
        pack = np.load(cache_path)
        feats = pack["features"].astype(np.float32); labels = pack["labels"].astype(np.int64)
        if feats.shape[0] == len(y) and np.array_equal(labels, y):
            print(f"  Cache OK: shape={feats.shape}")
            return feats, labels
        print("  Cache mismatch. Recomputing...")

    t0 = time.perf_counter()
    feats, labels = extract_dinov2_features(X, y, model_path, device,
                                            processor=processor, backbone=backbone)
    print(f"Feature extraction time: {time.perf_counter() - t0:.1f}s")
    np.savez(cache_path, features=feats, labels=labels)
    print(f"Saved feature cache: {cache_path}")
    return feats, labels


# ============================================================
# Classifier head + training
# ============================================================
class FeatureClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=1e-5):
        self.patience = patience; self.min_delta = min_delta
        self.best = float("inf"); self.counter = 0; self.best_state = None

    def step(self, loss, model):
        if loss < self.best - self.min_delta:
            self.best = loss; self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def train_head(X_tr, y_tr, X_va, y_va, input_dim, hidden_dim, num_classes, device):
    model = FeatureClassifier(input_dim, hidden_dim, num_classes).to(device)
    tr_loader = DataLoader(TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                                          torch.tensor(y_tr, dtype=torch.long)),
                           batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    va_loader = DataLoader(TensorDataset(torch.tensor(X_va, dtype=torch.float32),
                                          torch.tensor(y_va, dtype=torch.long)),
                           batch_size=BATCH_SIZE_EVAL, shuffle=False)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    ce = nn.CrossEntropyLoss()
    stopper = EarlyStopping()
    amp_enabled = USE_AMP and device.type == "cuda"
    scaler = torch.amp.GradScaler(device="cuda", enabled=amp_enabled)

    print(f"  Architecture: Frozen DINOv2 CLS({input_dim}) -> "
          f"Linear({input_dim}->{hidden_dim}) -> ReLU -> Linear({hidden_dim}->{num_classes})")
    print(f"  Device: {device}, AMP: {amp_enabled}")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, total_n = 0.0, 0
        for xb, yb in tr_loader:
            xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                logits = model(xb); loss = ce(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            total_loss += loss.item() * xb.size(0); total_n += xb.size(0)
        train_loss = total_loss / max(total_n, 1)

        model.eval()
        v_loss, v_n = 0.0, 0
        with torch.no_grad():
            for xb, yb in va_loader:
                xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
                with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                    logits = model(xb); loss = ce(logits, yb)
                v_loss += loss.item() * xb.size(0); v_n += xb.size(0)
        val_loss = v_loss / max(v_n, 1)

        if epoch == 1 or epoch % PRINT_EVERY == 0 or epoch == EPOCHS:
            print(f"    Epoch {epoch:>3d}/{EPOCHS} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

        if stopper.step(val_loss, model):
            print(f"    Early stopping at epoch {epoch}, best_val_loss={stopper.best:.6f}")
            break

    stopper.restore(model)
    model.cpu().eval()
    return model


# ============================================================
# Evaluation helpers (corruption sets)
# ============================================================
@torch.no_grad()
def _forward_np(model: nn.Module, X_np: np.ndarray, batch_size: int = 4096) -> np.ndarray:
    """Run a PyTorch model (on CPU) over numpy data in chunks and return logits as numpy."""
    model.eval()
    outs = []
    for i in range(0, len(X_np), batch_size):
        xb = torch.tensor(X_np[i:i + batch_size], dtype=torch.float32)
        outs.append(model(xb).cpu().numpy())
    return np.concatenate(outs, axis=0)


def evaluate_origin_on_corruptions(
    orig_model: nn.Module,
    corruption_feats_std: Dict[str, Tuple[np.ndarray, np.ndarray]],
) -> Dict[str, Dict[str, float]]:
    """Cache: evaluate Original head on all corruption sets once.

    Returns {corr_name: {"top1": ..., "top5": ..., "preds": argmax}}.
    `preds` is reused later to compute Quad-vs-Original agreement.
    """
    out: Dict[str, Dict[str, float]] = {}
    print("\nEvaluating Original ReLU head on corruption sets (one-time):")
    for corr_name, (X_corr, y_corr) in sorted(corruption_feats_std.items()):
        logits = _forward_np(orig_model, X_corr)
        preds = logits.argmax(axis=1)
        top1 = float(np.mean(preds == y_corr))
        top5 = topk_accuracy(logits, y_corr, k=5)
        out[corr_name] = {"top1": top1, "top5": top5, "preds": preds}
        print(f"  {corr_name:30s} | n={len(y_corr):5d} | orig top1/top5={top1:.4f}/{top5:.4f}")
    return out


def evaluate_quad_on_corruptions(
    quad_model: nn.Module,
    corruption_feats_std: Dict[str, Tuple[np.ndarray, np.ndarray]],
    orig_eval_cache: Dict[str, Dict[str, float]],
    out_csv_path: str,
    pool_label: str,
) -> List[Dict]:
    """Evaluate the Quad4FHE replaced_model on every corruption set.
    Reuses `orig_eval_cache` to compute orig_top1 / orig_top5 / agreement.
    """
    rows = []
    for corr_name, (X_corr, y_corr) in sorted(corruption_feats_std.items()):
        logits_quad = _forward_np(quad_model, X_corr)
        preds_quad = logits_quad.argmax(axis=1)
        quad_top1 = float(np.mean(preds_quad == y_corr))
        quad_top5 = topk_accuracy(logits_quad, y_corr, k=5)

        orig_info = orig_eval_cache.get(corr_name, {})
        orig_top1 = orig_info.get("top1", float("nan"))
        orig_top5 = orig_info.get("top5", float("nan"))
        orig_preds = orig_info.get("preds")
        agreement = float(np.mean(preds_quad == orig_preds)) if orig_preds is not None else float("nan")

        row = {
            "pool":       pool_label,
            "corruption": corr_name,
            "n":          int(len(y_corr)),
            "orig_top1":  orig_top1,
            "orig_top5":  orig_top5,
            "quad_top1":  quad_top1,
            "quad_top5":  quad_top5,
            "quad_vs_orig_agree": agreement,
        }
        rows.append(row)
        print(f"  {corr_name:30s} | n={len(y_corr):5d} | "
              f"orig top1/top5={orig_top1:.4f}/{orig_top5:.4f} | "
              f"quad top1/top5={quad_top1:.4f}/{quad_top5:.4f} | "
              f"agree={agreement:.4f}")

    if rows:
        pd.DataFrame(rows).to_csv(out_csv_path, index=False)
        print(f"  Saved corruption robustness CSV: {out_csv_path}")
    return rows


# ============================================================
# Main
# ============================================================
def main():
    set_seed(SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---------------- Step 1: Load + 4-way split ----------------
    separator("Step 1: Load Stanford Cars and create 4-way split "
              f"(train {TRAIN_SIZE*100:.0f}% / val {VAL_SIZE*100:.0f}% / "
              f"test {TEST_SIZE*100:.0f}% / pool_max {POOL_MAX_SIZE*100:.0f}%)")
    X, y = load_stanford_cars_merged(STANFORDCARS_DATA_ROOT)
    if STANFORDCARS_CLASS_NAMES:
        sample_names = {k: v for k, v in list(STANFORDCARS_CLASS_NAMES.items())[:5]}
        print(f"  Class names (first 5): {sample_names} ...")
    total_size = len(X)
    split_idx = split_4way_indices(y)
    for key in ["train", "val", "pool_max", "test"]:
        idx = split_idx[key]
        pct = len(idx) / total_size * 100.0
        print(f"  {key:10s}: n={len(idx):6d} ({pct:5.2f}%)")

    print("\n  Pool subset sizes (per SURROGATE_FRACTIONS):")
    for f in SURROGATE_FRACTIONS:
        n_target = int(round(total_size * f))
        n_actual = min(n_target, len(split_idx["pool_max"]))
        note = ""
        if n_target < NUM_CLASSES:
            note = f"  [!] target < {NUM_CLASSES} classes -> non-stratified fallback (likely skipped by MIN_SAMPLES_PER_CLASS)"
        print(f"    fraction={f*100:5.1f}% of total -> target n={n_target:6d}, actual n={n_actual:6d}{note}")

    # ---------------- Step 1b: Load corruption splits ----------------
    sub_separator("Step 1b: Load corruption splits")
    corruption_images = load_corruption_splits(STANFORDCARS_DATA_ROOT)

    # ---------------- Step 2: DINOv2 features (train+test) ----------------
    separator("Step 2: Extract / load frozen DINOv2 features (train+test)")
    feats_all, labels_all = ensure_feature_cache(X, y, DINOV2_LOCAL_PATH, FEATURE_CACHE, device)
    print(f"  Feature matrix shape: {feats_all.shape}")

    # ---------------- Step 2b: DINOv2 features (corruption splits) ----------------
    corruption_feats_raw: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
    if corruption_images:
        sub_separator("Step 2b: Extract / load DINOv2 features for corruption splits")
        cache_dir = os.path.dirname(FEATURE_CACHE)

        need_backbone = False
        for corr_name in corruption_images:
            corr_cache = os.path.join(cache_dir, f"dinov2_base_stanfordcars_{corr_name}_features.npz")
            if not os.path.exists(corr_cache):
                need_backbone = True
                break

        processor_shared, backbone_shared = None, None
        if need_backbone:
            print(f"  Loading DINOv2 backbone once from {DINOV2_LOCAL_PATH} (reused across corruptions) ...")
            processor_shared = AutoImageProcessor.from_pretrained(DINOV2_LOCAL_PATH)
            backbone_shared = AutoModel.from_pretrained(DINOV2_LOCAL_PATH).to(device)
            backbone_shared.eval()

        try:
            for corr_name, (X_corr, y_corr) in sorted(corruption_images.items()):
                corr_cache = os.path.join(cache_dir, f"dinov2_base_stanfordcars_{corr_name}_features.npz")
                corr_feats, corr_labels = ensure_feature_cache(
                    X_corr, y_corr, DINOV2_LOCAL_PATH, corr_cache, device,
                    processor=processor_shared, backbone=backbone_shared,
                )
                corruption_feats_raw[corr_name] = (corr_feats, corr_labels)
                print(f"  {corr_name}: features shape = {corr_feats.shape}")
        finally:
            if backbone_shared is not None:
                del backbone_shared, processor_shared
                gc.collect()
                if device.type == "cuda":
                    torch.cuda.empty_cache()
                print("  DINOv2 backbone released from memory.")

    # Release raw images from memory
    del X
    for _k in list(corruption_images.keys()):
        del corruption_images[_k]
    del corruption_images
    gc.collect()

    X_train    = feats_all[split_idx["train"]];    y_train    = labels_all[split_idx["train"]]
    X_val      = feats_all[split_idx["val"]];      y_val      = labels_all[split_idx["val"]]
    X_pool_max = feats_all[split_idx["pool_max"]]; y_pool_max = labels_all[split_idx["pool_max"]]
    X_test     = feats_all[split_idx["test"]];     y_test     = labels_all[split_idx["test"]]

    # Fit scaler on train only
    scaler = StandardScaler().fit(X_train)
    X_train_std    = scaler.transform(X_train).astype(np.float64)
    X_val_std      = scaler.transform(X_val).astype(np.float64)
    X_pool_max_std = scaler.transform(X_pool_max).astype(np.float64)
    X_test_std     = scaler.transform(X_test).astype(np.float64)
    input_dim = X_train_std.shape[1]
    print(f"  DINOv2 feature dim (CLS token) = {input_dim}")

    # Apply scaler to corruption features
    corruption_feats_std: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
    for corr_name, (corr_f, corr_l) in corruption_feats_raw.items():
        corruption_feats_std[corr_name] = (scaler.transform(corr_f).astype(np.float64), corr_l)
    del corruption_feats_raw
    if corruption_feats_std:
        print(f"  Corruption splits standardised: {list(corruption_feats_std.keys())}")

    # ---------------- Step 3: Train head once ----------------
    separator(f"Step 3: Train classifier head (hidden_dim={HIDDEN_DIM})")
    set_seed(SEED)
    model = train_head(X_train_std, y_train, X_val_std, y_val,
                       input_dim, HIDDEN_DIM, NUM_CLASSES, device)

    with torch.no_grad():
        orig_test_logits = model(torch.tensor(X_test_std, dtype=torch.float32)).cpu().numpy()
    orig_top1 = float(np.mean(orig_test_logits.argmax(axis=1) == y_test))
    orig_top5 = topk_accuracy(orig_test_logits, y_test, k=5)
    print(f"\n  Original head test top-1 ACC = {orig_top1:.4f}")
    print(f"  Original head test top-5 ACC = {orig_top5:.4f}")

    # Original head on corruption sets (compute ONCE; reused across all pool fractions)
    orig_corr_cache: Dict[str, Dict[str, float]] = {}
    if corruption_feats_std:
        separator("Step 3b: Evaluate Original head on corruption sets (one-time, reused)")
        orig_corr_cache = evaluate_origin_on_corruptions(model, corruption_feats_std)

    # ---------------- Step 4: Loop over pool fractions ----------------
    separator("Step 4: quad4fhe.replace() per pool fraction")
    all_pool_results = {}
    all_pool_top5    = {}
    all_pool_fit_sizes = {}
    all_pool_skips = {}

    payload = make_experiment_payload(
        dataset=DATASET_NAME,
        experiment=EXPERIMENT_NAME,
        seed=SEED,
        dataset_info={
            "input_dim": int(input_dim),
            "num_classes": int(NUM_CLASSES),
            "splits": {
                "source": "fixed_stratified_4way_split_from_merged_dataset",
                "train": int(len(X_train_std)),
                "val": int(len(X_val_std)),
                "pool_max": int(len(X_pool_max_std)),
                "test": int(len(X_test_std)),
            },
            "dinov2_model": DINOV2_LOCAL_PATH,
            "feature_cache": FEATURE_CACHE,
            "data_root": (
                globals().get("CIFAR_DATA_ROOT")
                or globals().get("FGVC_DATA_ROOT")
                or globals().get("STANFORDCARS_DATA_ROOT")
                or globals().get("TINY_DATA_ROOT")
            ),
            "frozen_backbone": True,
            "encrypted_component": "classification_head_only",
            "corruption_splits": list(corruption_feats_std.keys()) if "corruption_feats_std" in locals() else [],
        },
        config={
            "hidden_dim": HIDDEN_DIM,
            "epochs": EPOCHS,
            "lr": LR,
            "batch_size_train": BATCH_SIZE_TRAIN,
            "batch_size_eval": BATCH_SIZE_EVAL,
            "weight_decay": WEIGHT_DECAY,
            "patience": PATIENCE,
            "train_size": TRAIN_SIZE,
            "val_size": VAL_SIZE,
            "test_size": TEST_SIZE,
            "pool_max_size": POOL_MAX_SIZE,
            "surrogate_fractions": SURROGATE_FRACTIONS,
            "min_pool_samples": MIN_POOL_SAMPLES,
            "min_samples_per_class": MIN_SAMPLES_PER_CLASS,
            "image_resize": globals().get("IMG_RESIZE"),
            "use_amp": globals().get("USE_AMP"),
            "feature_batch_size": globals().get("BATCH_SIZE_FEATURE"),
            "baselines": BASELINES,
        },
        source_script=f"{DATASET_NAME}_{EXPERIMENT_NAME}_autosave.ipynb",
        log_file=LOG_PATH,
    )
    all_corruption_rows = []

    for fraction in SURROGATE_FRACTIONS:
        pool_label = f"{int(fraction*100):02d}%"
        print("\n" + "-" * 78)
        print(f"POOL = {fraction*100:.0f}%")
        print("-" * 78)

        sub_idx = subsample_pool(y_pool_max, fraction, total_size, seed=SEED)
        X_rep_fit = X_pool_max_std[sub_idx]
        y_rep_fit = y_pool_max[sub_idx]

        if len(X_rep_fit) < MIN_POOL_SAMPLES:
            print(f"  [SKIPPED] rep_fit too small (n={len(X_rep_fit)} < {MIN_POOL_SAMPLES})")
            all_pool_results[fraction] = None; all_pool_top5[fraction] = None
            all_pool_skips[fraction] = "rep_fit_too_small"
            continue

        counts = np.bincount(y_rep_fit, minlength=NUM_CLASSES)
        missing = [i for i, c in enumerate(counts) if c < MIN_SAMPLES_PER_CLASS]
        if missing:
            print(f"  [SKIPPED] {len(missing)} classes have < {MIN_SAMPLES_PER_CLASS} samples "
                  f"(first 10 classes: {missing[:10]}{'...' if len(missing)>10 else ''})")
            print(f"  Per-class counts: min={counts.min()}, max={counts.max()}, mean={counts.mean():.1f}")
            all_pool_results[fraction] = None; all_pool_top5[fraction] = None
            all_pool_skips[fraction] = "missing_class_or_too_few_samples_per_class"
            continue

        print(f"  rep_fit: n={len(X_rep_fit)}, per-class counts: "
              f"min={counts.min()}, max={counts.max()}, mean={counts.mean():.1f}")

        result = quad4fhe.replace(
            task              = "multiclass",
            model             = model,
            hidden_layer      = "fc1",
            output_layer      = "fc2",
            activation        = "relu",
            fit_data          = (X_rep_fit, y_rep_fit),
            test_data         = (X_test_std, y_test),
            baselines         = BASELINES,
            export_he_dir     = f"he_artifacts_dinov2_stanfordcars_pool_{int(fraction*100):02d}",
            use_cutting_plane = True,
            use_osqp_warmstart= False,
            verbose           = True,
            seed              = SEED,
        )
        all_pool_results[fraction] = result

        rm = result.replaced_model.eval()
        with torch.no_grad():
            logits_quad = rm(torch.tensor(X_test_std, dtype=torch.float32)).cpu().numpy()
        quad_top5 = topk_accuracy(logits_quad, y_test, k=5)
        all_pool_top5[fraction] = quad_top5


        all_pool_fit_sizes[fraction] = len(X_rep_fit)
        fit_diag = quad_report_diagnostics(result, "fit", n_expected=len(X_rep_fit))
        test_diag = quad_report_diagnostics(result, "test", n_expected=len(X_test_std))
        run_record = build_quad4fhe_run_record(
            result=result,
            key=f"pool={int(fraction*100):02d}%",
            hidden_dim=HIDDEN_DIM,
            fit_n=len(X_rep_fit),
            test_n=len(X_test_std),
            pool_fraction=fraction,
            rep_fit_size=len(X_rep_fit),
            extra={"pool_label": f"{int(fraction*100):02d}%"},
        )
        run_record["original"]["test_top5_acc"] = orig_top5
        run_record["quad4fhe"]["test_top5_acc"] = quad_top5
        payload["runs"].append(run_record)
        save_json(payload, JSON_PATH)
        print(f"  calib agreement={fit_diag.get('agreement')} "
              f"mismatches={fit_diag.get('mismatch_count')} "
              f"exact_preserved={fit_diag.get('exact_preserved')}")
        print(f"  test agreement={test_diag.get('agreement')} "
              f"mismatches={test_diag.get('mismatch_count')}")

        print(f"  method_used={result.method_used}, "
              f"hard_feasible={result.feasible}, "
              f"quant_decimals={result.quant_decimals}")

        # Save per-fraction report
        frac_dir = os.path.join(OUTPUT_DIR, f"pool_{int(fraction*100):02d}")
        os.makedirs(frac_dir, exist_ok=True)
        if result.report_vs_truth is not None:
            result.report_vs_truth.to_csv(os.path.join(frac_dir, "report_vs_truth.csv"), index=False)
            row = result.report_vs_truth[
                (result.report_vs_truth["Method"] == "Quad4FHE") &
                (result.report_vs_truth["Split"]  == "test")
            ]
            if len(row) > 0:
                q = row.iloc[0]
                print(f"  Quad4FHE test top-1 ACC={q['ACC']:.4f}  top-5 ACC={quad_top5:.4f}  "
                      f"MacroF1={q.get('MacroF1', float('nan')):.4f}  "
                      f"Agreement={q.get('Agreement', float('nan')):.4f}")

        # ---------------- Corruption robustness evaluation ----------------
        if corruption_feats_std:
            print(f"\n  [pool={pool_label}] Robustness on corruption sets:")
            corr_rows = evaluate_quad_on_corruptions(
                quad_model=rm,
                corruption_feats_std=corruption_feats_std,
                orig_eval_cache=orig_corr_cache,
                out_csv_path=os.path.join(frac_dir, "robustness_corruptions.csv"),
                pool_label=pool_label,
            )
            all_corruption_rows.extend(corr_rows)

    # ---------------- Step 5: Cross-pool comparison ----------------
    separator("Step 5: Cross-pool comparison")

    pool_labels = [f"{int(f*100):02d}%" for f in SURROGATE_FRACTIONS]

    first = next((r for r in all_pool_results.values() if r is not None), None)
    if first is not None and first.report_vs_truth is not None:
        available_cols = list(first.report_vs_truth.columns)
    else:
        available_cols = []
    METRIC_KEYS = [c for c in ["ACC", "MacroF1", "MacroPrec", "MacroRec", "AUC_ovr", "Agreement"]
                   if c in available_cols]

    method_order = ["Original", "Quad4FHE"] + BASELINES

    def extract_test_row(res, method):
        if res is None or res.report_vs_truth is None: return None
        df = res.report_vs_truth
        r = df[(df["Method"] == method) & (df["Split"] == "test")]
        return r.iloc[0].to_dict() if len(r) > 0 else None

    def _build_matrix(metric):
        data = {}
        for f, lbl in zip(SURROGATE_FRACTIONS, pool_labels):
            col = {}
            for m in method_order:
                row = extract_test_row(all_pool_results[f], m)
                col[m] = row[metric] if row is not None and metric in row else float("nan")
            data[f"pool={lbl}"] = col
        return pd.DataFrame(data)

    cross_pool_tables = {}
    for metric in METRIC_KEYS:
        title = ("Test Agreement (with original model / pseudo-labels)"
                 if metric == "Agreement"
                 else f"Test {metric} (vs TRUE labels)")
        print(f"\n--- Table: {title} ---")
        df_m = _build_matrix(metric)
        cross_pool_tables[metric] = df_m.to_dict()
        with pd.option_context("display.float_format", "{:.4f}".format):
            print(df_m.to_string())

    # Quad4FHE top-5 table
    print(f"\n--- Table: Test Top-5 ACC (Quad4FHE only) ---")
    top5_quad = {f"pool={lbl}": all_pool_top5.get(f, float("nan"))
                 for f, lbl in zip(SURROGATE_FRACTIONS, pool_labels)}
    top5_orig = {f"pool={lbl}": orig_top5 for _, lbl in zip(SURROGATE_FRACTIONS, pool_labels)}
    df_top5 = pd.DataFrame({"Original": top5_orig, "Quad4FHE": top5_quad}).T
    with pd.option_context("display.float_format", "{:.4f}".format):
        print(df_top5.to_string())

    # Meta table
    print("\n--- Meta Table ---")
    meta_rows = []
    for f, lbl in zip(SURROGATE_FRACTIONS, pool_labels):
        r = all_pool_results[f]
        if r is None:
            row = {
                "pool": lbl,
                "status": "SKIPPED",
                "skip_reason": all_pool_skips.get(f, "skipped_by_pool_guard"),
                "pool_fraction": f,
                "hidden_dim": HIDDEN_DIM,
            }
            meta_rows.append(row)
            payload["runs"].append({
                "key": f"pool={lbl}",
                "hidden_dim": HIDDEN_DIM,
                "pool_fraction": f,
                "status": "skipped",
                "skip_reason": row["skip_reason"],
                "quad4fhe": {"status": "skipped", "skip_reason": row["skip_reason"]},
            })
            continue
        fit_diag = quad_report_diagnostics(r, "fit", n_expected=all_pool_fit_sizes.get(f))
        test_diag = quad_report_diagnostics(r, "test", n_expected=len(X_test_std))
        solver_diag = quad_solver_diagnostics(r)
        meta_rows.append({
            "pool":             lbl,
            "pool_fraction":    f,
            "hidden_dim":       HIDDEN_DIM,
            "method_used":      r.method_used,
            "hard_feasible":    r.feasible,
            "empirical_margin": round(r.empirical_margin, 6),
            "norm_margin":      round(r.normalized_margin, 6),
            "quant_decimals":   r.quant_decimals,
            "calib_n":          fit_diag.get("n"),
            "calib_agreement":  fit_diag.get("agreement"),
            "calib_mismatch":   fit_diag.get("mismatch_count"),
            "exact_preserved":  fit_diag.get("exact_preserved"),
            "test_agreement":   test_diag.get("agreement"),
            "test_mismatch":    test_diag.get("mismatch_count"),
            "calib_test_gap":   (None if fit_diag.get("agreement") is None or test_diag.get("agreement") is None
                                  else fit_diag.get("agreement") - test_diag.get("agreement")),
            "num_pairwise_constraints": solver_diag.get("num_pairwise_constraints"),
            "min_pairwise_margin": solver_diag.get("min_pairwise_margin"),
            "normalized_min_pairwise_margin": solver_diag.get("normalized_min_pairwise_margin"),
            "slack_pos":        solver_diag.get("slack_positive_count"),
            "sum_slack":        solver_diag.get("sum_slack"),
            "mean_slack":       solver_diag.get("mean_slack"),
            "max_slack":        solver_diag.get("max_slack"),
            "pairwise_slack_pos": solver_diag.get("pairwise_slack_positive_count"),
            "pairwise_sum_slack": solver_diag.get("pairwise_sum_slack"),
            "pairwise_mean_slack": (None if solver_diag.get("pairwise_sum_slack") is None or solver_diag.get("num_pairwise_constraints") in (None, 0)
                                     else solver_diag.get("pairwise_sum_slack") / solver_diag.get("num_pairwise_constraints")),
            "pairwise_max_slack": solver_diag.get("pairwise_max_slack"),
            "selected_C":       solver_diag.get("selected_C"),
            "selected_mu":      solver_diag.get("selected_mu"),
            "constraint_version": getattr(r, "constraint_version", None),
            "quad_top5":        round(all_pool_top5.get(f, float("nan")), 4) if all_pool_top5.get(f) is not None else None,
            "he_export_dir":    os.path.basename(r.he_export_dir) if r.he_export_dir else None,
        })
    df_meta = pd.DataFrame(meta_rows)
    print(df_meta.to_string(index=False))
    df_meta.to_csv(os.path.join(OUTPUT_DIR, "meta_all_pools.csv"), index=False)
    print(f"\nSaved meta CSV: {OUTPUT_DIR}/meta_all_pools.csv")


    payload["cross_pool_tables"] = cross_pool_tables
    payload["meta_table"] = meta_rows
    if "all_corruption_rows" in locals():
        payload["corruption_summary"] = all_corruption_rows
    save_json(payload, JSON_PATH)
    save_csv(meta_rows, META_CSV_PATH)
    write_combined_dataset_json(DATASET_NAME, root_dir=OUTPUT_DIR.parent.parent)

    # ---------------- Step 5b: Combined corruption summary ----------------
    if all_corruption_rows:
        separator("Step 5b: Cross-pool corruption robustness summary")
        df_corr = pd.DataFrame(all_corruption_rows)
        with pd.option_context("display.float_format", "{:.4f}".format, "display.max_columns", None):
            print(df_corr.to_string(index=False))
        corr_csv = os.path.join(OUTPUT_DIR, "robustness_corruptions_all_pools.csv")
        df_corr.to_csv(corr_csv, index=False)
        print(f"\nSaved combined corruption summary: {corr_csv}")

        # Pivot tables: rows=corruption, cols=pool
        try:
            pivot_quad = df_corr.pivot(index="corruption", columns="pool", values="quad_top1")
            pivot_orig = df_corr.pivot(index="corruption", columns="pool", values="orig_top1")
            print("\n--- Pivot: Original head top-1 ACC (rows=corruption, cols=pool) ---")
            with pd.option_context("display.float_format", "{:.4f}".format):
                print(pivot_orig.to_string())
            print("\n--- Pivot: Quad4FHE top-1 ACC (rows=corruption, cols=pool) ---")
            with pd.option_context("display.float_format", "{:.4f}".format):
                print(pivot_quad.to_string())
        except Exception as e:
            print(f"  (Pivot display skipped: {e})")

    separator("Done")


if __name__ == "__main__":
    try:
        with tee_stdout_stderr(LOG_PATH):
            main()
    except Exception:
        traceback.print_exc()
        sys.stdout.flush(); sys.stderr.flush()
        raise

[autosave] stdout/stderr log -> ../results/Dinov2_StanfordCars/smallpool/Dinov2_StanfordCars_smallpool_result.txt

Step 1: Load Stanford Cars and create 4-way split (train 50% / val 10% / test 20% / pool_max 20%)
Loading Stanford Cars from local parquet files in: ./data_StanfordCars
  Train files: ['train-00000-of-00002.parquet', 'train-00001-of-00002.parquet']
  Test  files: ['test-00000-of-00002.parquet', 'test-00001-of-00002.parquet']
  [train] loaded 1/8144
  [train] loaded 2000/8144
  [train] loaded 4000/8144
  [train] loaded 6000/8144
  [train] loaded 8000/8144
  [train] loaded 8144/8144
  Loaded 8144 train images
  [test] loaded 1/8041
  [test] loaded 2000/8041
  [test] loaded 4000/8041
  [test] loaded 6000/8041
  [test] loaded 8000/8041
  [test] loaded 8041/8041
  Loaded 8041 test images
  Total images (train + test): 16185, shape: (16185, 224, 224, 3), classes: 196
  Class names (first 5): {0: 'AM General Hummer SUV 2000', 1: 'Acura RL Sedan 2012', 2: 'Acura TL Sedan 2012', 3: